In [2]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns  

In [3]:
# 1. Load dataset
df = pd.read_excel('shipwreckdatabase.xlsx')
# Clean column names
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

In [4]:
 # Ensure description is string
df['description'] = df['description'].astype(str)
print(f"Dataset shape: {df.shape}")
print("Classification labels:", df['classification'].unique())

Dataset shape: (28234, 7)
Classification labels: ['Schooner' 'Sloop' 'Iron Steamship' 'Brig' 'Unknown' 'Lugger' 'Snow'
 'Steamship' 'Smack' 'Yawl' 'Steel Steamship' 'Cutter' 'Ketch'
 'Brigantine' 'Lighter' 'Barque' 'Yacht' 'Ship' 'Fishing boat'
 'Crane Vessel' 'Tender' 'Barquentine' 'Collier' 'Dutch Galliot'
 'Screw Steamer' 'Flat' 'Bark' 'Boat' 'Trader' 'Carvel' 'Lugsail'
 'Galliot' 'Dandy' 'Clinker' 'Iron Paddle Steam Tug' 'Sailing Ship'
 'Merchant Vessel' 'Pleasure Boat' 'Trawler' 'Pilot boat' 'Barge'
 'Admiralty/ Decoy Ship' 'Steam Trawler' 'Submarine' 'Sailing Boat'
 'Paddler Steamer' 'Motor Boat' 'Packet boat' 'Wherry' 'Coaster' 'Clipper'
 'Cruiser' 'Vessel' 'Iron Steam Yacht' 'Smack Yacht' 'Hacker' 'Jigger'
 'Tug' 'Wooden Steam Tug' 'Frigate' 'Hobble' 'Steam Tug' 'Stone hacker'
 'Gabbard' 'West Indiaman' 'Skiff' "Man O' War" 'Anti-submarine Drifter'
 'Light Ship' 'Steam Drifter' 'Fishing drifter' 'Lifeboat'
 'Coast Guard Cruiser' 'Iron steam tug' 'Motor Fishing Vessel'
 'Steam P

In [5]:
# Drop rows with missing descriptions
df.dropna(subset=['description'], inplace=True)

In [6]:
print(df[['wreck_name','classification', 'place_of_loss', 'date_of_loss', 'description']].head())

           wreck_name  classification  \
0               Actur        Schooner   
1         Anne McLeod        Schooner   
2  Barbara & Jannette           Sloop   
3            Bee (SS)  Iron Steamship   
4               Bloom        Schooner   

                                       place_of_loss date_of_loss  \
0  Carlingford Bar to Greencastle pier, Carlingfo...   28/12/1894   
1               Carlingford, near, Carlingford Lough   22/01/1862   
2  Carlingford Bar, outside, on a rock, Carlingfo...   18/02/1824   
3      Carlingford Lough entrance/ Helly Hunter Rock   19/10/1888   
4                 Carlingford Bar, Carlingford Lough   23/10/1838   

                                         description  
0  Schooner of Dublin, stranded en route to Cunni...  
1  71-ton, 6-year-old schooner. Captain was Darby...  
2  Sloop en route from Irvine to Dundalk, struck ...  
3  88-ton, 18-year-old iron steamship. Owned by ‘...  
4  Schooner valued at £600, became a total wreck,...  


In [7]:
# 2. Extract cause from description
def extract_cause(desc):
    desc = desc.lower()
    if 'torpedo' in desc:
        return 'torpedoed'
    elif 'mine' in desc:
        return 'mined'
    elif 'ran aground' in desc or 'rock' in desc:
        return 'ran aground'
    elif 'storm' in desc or 'weather' in desc or 'typhoon' in desc or 'cyclone' in desc:
        return 'bad weather'
    elif 'collision' in desc or 'collided' in desc:
        return 'collision'
    elif 'fire' in desc or 'explosion' in desc or 'burn' in desc:
        return 'fire/explosion'
    elif 'sank' in desc or 'sinking' in desc or 'water leakage' in desc or 'flood' in desc:
        return 'sank'
    elif 'naval battle' in desc or 'air raid' in desc or 'gunfire' in desc:
        return 'naval battle'
    elif 'ice' in desc:
        return 'iceberg wreck'
    elif 'scuttle' in desc:
        return 'scuttle'
    elif 'wreck' in desc:
        return 'wrecked'
    elif 'stranded' in desc or 'abandon' in desc or 'foundered' in desc:
        return 'foundered'
    else:
        return 'unknown reasons'



In [8]:
df['cause'] = df['description'].apply(extract_cause)

In [9]:
print("\nLabel distribution (before cleaning):")
print(df['cause'].value_counts())


Label distribution (before cleaning):
cause
unknown reasons    14520
foundered           5191
wrecked             3104
torpedoed            976
sank                 904
ran aground          837
fire/explosion       626
bad weather          592
collision            548
mined                338
naval battle         303
scuttle              244
iceberg wreck         51
Name: count, dtype: int64


In [10]:
#  Remove unknown causes globally
df = df[df['cause'] != 'unknown reasons'].copy()


In [11]:
print("\nLabel distribution (after removing unknown):")
print(df['cause'].value_counts())



Label distribution (after removing unknown):
cause
foundered         5191
wrecked           3104
torpedoed          976
sank               904
ran aground        837
fire/explosion     626
bad weather        592
collision          548
mined              338
naval battle       303
scuttle            244
iceberg wreck       51
Name: count, dtype: int64


In [12]:
df

,wreck_name,classification,place_of_loss,date_of_loss,description,unnamed:_5,unnamed:_6,cause
0,Actur,Schooner,"Carlingford Bar to Greencastle pier, Carlingfo...",28/12/1894,"Schooner of Dublin, stranded en route to Cunni...",NaN,NaN,foundered
1,Anne McLeod,Schooner,"Carlingford, near, Carlingford Lough",22/01/1862,"71-ton, 6-year-old schooner. Captain was Darby...",NaN,NaN,foundered
2,Barbara & Jannette,Sloop,"Carlingford Bar, outside, on a rock, Carlingfo...",18/02/1824,"Sloop en route from Irvine to Dundalk, struck ...",NaN,NaN,ran aground
3,Bee (SS),Iron Steamship,Carlingford Lough entrance/ Helly Hunter Rock,19/10/1888,"88-ton, 18-year-old iron steamship. Owned by ‘...",NaN,NaN,ran aground
4,Bloom,Schooner,"Carlingford Bar, Carlingford Lough",23/10/1838,"Schooner valued at £600, became a total wreck,...",NaN,NaN,wrecked
...,...,...,...,...,...,...,...,...
28222,USSÂ Grenadier,submarine,"Phuket, Thailand",1943-04-22 00:00:00,"A Tambor-class submarine scuttled off Phuket, ...",NaN,NaN,scuttle
28228,MyÅkÅ,cruiser,Port Klang,1946-06-08 00:00:00,A MyÅkÅ-class cruiser that was scuttled near...,NaN,NaN,scuttle
28229,Sovereign of the Seas,Clipper,Pyramid Shoal.,6 August 1859,A clipper that was wrecked on the Pyramid Shoal.,NaN,NaN,wrecked
28231,U-181,U-Boat,Port Klang,1946-02-12 00:00:00,A Type IXD2 U-boat that was scuttled near Port...,NaN,NaN,scuttle


In [13]:
df=df.drop_duplicates()

In [14]:
df

,wreck_name,classification,place_of_loss,date_of_loss,description,unnamed:_5,unnamed:_6,cause
0,Actur,Schooner,"Carlingford Bar to Greencastle pier, Carlingfo...",28/12/1894,"Schooner of Dublin, stranded en route to Cunni...",NaN,NaN,foundered
1,Anne McLeod,Schooner,"Carlingford, near, Carlingford Lough",22/01/1862,"71-ton, 6-year-old schooner. Captain was Darby...",NaN,NaN,foundered
2,Barbara & Jannette,Sloop,"Carlingford Bar, outside, on a rock, Carlingfo...",18/02/1824,"Sloop en route from Irvine to Dundalk, struck ...",NaN,NaN,ran aground
3,Bee (SS),Iron Steamship,Carlingford Lough entrance/ Helly Hunter Rock,19/10/1888,"88-ton, 18-year-old iron steamship. Owned by ‘...",NaN,NaN,ran aground
4,Bloom,Schooner,"Carlingford Bar, Carlingford Lough",23/10/1838,"Schooner valued at £600, became a total wreck,...",NaN,NaN,wrecked
...,...,...,...,...,...,...,...,...
28222,USSÂ Grenadier,submarine,"Phuket, Thailand",1943-04-22 00:00:00,"A Tambor-class submarine scuttled off Phuket, ...",NaN,NaN,scuttle
28228,MyÅkÅ,cruiser,Port Klang,1946-06-08 00:00:00,A MyÅkÅ-class cruiser that was scuttled near...,NaN,NaN,scuttle
28229,Sovereign of the Seas,Clipper,Pyramid Shoal.,6 August 1859,A clipper that was wrecked on the Pyramid Shoal.,NaN,NaN,wrecked
28231,U-181,U-Boat,Port Klang,1946-02-12 00:00:00,A Type IXD2 U-boat that was scuttled near Port...,NaN,NaN,scuttle


In [15]:
df = df.drop(columns=["unnamed:_5", "unnamed:_6"],axis=1)


In [16]:
df

,wreck_name,classification,place_of_loss,date_of_loss,description,cause
0,Actur,Schooner,"Carlingford Bar to Greencastle pier, Carlingfo...",28/12/1894,"Schooner of Dublin, stranded en route to Cunni...",foundered
1,Anne McLeod,Schooner,"Carlingford, near, Carlingford Lough",22/01/1862,"71-ton, 6-year-old schooner. Captain was Darby...",foundered
2,Barbara & Jannette,Sloop,"Carlingford Bar, outside, on a rock, Carlingfo...",18/02/1824,"Sloop en route from Irvine to Dundalk, struck ...",ran aground
3,Bee (SS),Iron Steamship,Carlingford Lough entrance/ Helly Hunter Rock,19/10/1888,"88-ton, 18-year-old iron steamship. Owned by ‘...",ran aground
4,Bloom,Schooner,"Carlingford Bar, Carlingford Lough",23/10/1838,"Schooner valued at £600, became a total wreck,...",wrecked
...,...,...,...,...,...,...
28222,USSÂ Grenadier,submarine,"Phuket, Thailand",1943-04-22 00:00:00,"A Tambor-class submarine scuttled off Phuket, ...",scuttle
28228,MyÅkÅ,cruiser,Port Klang,1946-06-08 00:00:00,A MyÅkÅ-class cruiser that was scuttled near...,scuttle
28229,Sovereign of the Seas,Clipper,Pyramid Shoal.,6 August 1859,A clipper that was wrecked on the Pyramid Shoal.,wrecked
28231,U-181,U-Boat,Port Klang,1946-02-12 00:00:00,A Type IXD2 U-boat that was scuttled near Port...,scuttle


In [17]:
#Learning Method
from sklearn.ensemble import RandomForestClassifier
X=df['description']
y=df['cause']

In [18]:
vectorizer= TfidfVectorizer(stop_words='english', max_features=3000)
X_vec = vectorizer.fit_transform(X)

In [19]:
X_train,X_test,y_train,y_test=train_test_split(X_vec,y,test_size=0.2,random_state=42,stratify=y)

In [20]:
model=RandomForestClassifier()
model.fit(X_train,y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [23]:
y_pred=model.predict(X_test)

In [24]:
print("\nClassification Report:")
print(classification_report(y_test, y_pred))




Classification Report:
                precision    recall  f1-score   support

   bad weather       0.96      0.86      0.91       102
     collision       0.99      0.93      0.96       108
fire/explosion       0.99      0.87      0.92       120
     foundered       0.99      1.00      0.99      1024
 iceberg wreck       0.00      0.00      0.00        10
         mined       0.78      0.71      0.75        66
  naval battle       1.00      1.00      1.00        56
   ran aground       0.97      0.87      0.92       162
          sank       0.91      0.96      0.93       180
       scuttle       0.87      0.98      0.92        49
     torpedoed       0.98      0.98      0.98       190
       wrecked       0.94      1.00      0.97       613

      accuracy                           0.96      2680
     macro avg       0.87      0.85      0.85      2680
  weighted avg       0.96      0.96      0.96      2680



In [25]:
from sklearn.metrics import accuracy_score

In [26]:
print("Accuracy Score:\n")
print(accuracy_score(y_test,y_pred))

Accuracy Score:

0.9619402985074627


In [27]:
from sklearn.tree import DecisionTreeClassifier

In [28]:
model=DecisionTreeClassifier()

In [29]:
model.fit(X_train,y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [30]:
y_pred=model.predict(X_test)

In [31]:
print("Classification Report :\n")
print(classification_report(y_test,y_pred))

Classification Report :

                precision    recall  f1-score   support

   bad weather       0.97      0.92      0.94       102
     collision       0.99      0.98      0.99       108
fire/explosion       0.98      0.88      0.93       120
     foundered       0.98      1.00      0.99      1024
 iceberg wreck       0.12      0.10      0.11        10
         mined       0.72      0.74      0.73        66
  naval battle       0.98      0.98      0.98        56
   ran aground       0.93      0.93      0.93       162
          sank       0.92      0.94      0.93       180
       scuttle       0.92      0.94      0.93        49
     torpedoed       0.98      0.99      0.99       190
       wrecked       0.98      0.98      0.98       613

      accuracy                           0.96      2680
     macro avg       0.87      0.86      0.87      2680
  weighted avg       0.96      0.96      0.96      2680



In [32]:
print("Accuracy Score:\n")
print(accuracy_score(y_test,y_pred))

Accuracy Score:

0.9649253731343284


In [33]:
model=LogisticRegression(max_iter=1000)

In [34]:
model.fit(X_train,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [35]:
y_pred=model.predict(X_test)

In [36]:
print("Classification Report: \n")
print(classification_report(y_test,y_pred))

Classification Report: 

                precision    recall  f1-score   support

   bad weather       0.97      0.82      0.89       102
     collision       0.99      0.93      0.96       108
fire/explosion       0.99      0.85      0.91       120
     foundered       0.99      1.00      0.99      1024
 iceberg wreck       0.00      0.00      0.00        10
         mined       0.80      0.71      0.75        66
  naval battle       1.00      0.98      0.99        56
   ran aground       0.97      0.88      0.92       162
          sank       0.90      0.95      0.93       180
       scuttle       0.87      0.96      0.91        49
     torpedoed       0.99      0.97      0.98       190
       wrecked       0.92      1.00      0.96       613

      accuracy                           0.96      2680
     macro avg       0.86      0.84      0.85      2680
  weighted avg       0.95      0.96      0.95      2680



C:\Users\drapa\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\drapa\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\drapa\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [37]:
print("Accuracy Score\n")
print(accuracy_score(y_test,y_pred))

Accuracy Score

0.957089552238806


In [38]:
model=SVC(kernel='linear')
model.fit(X_train,y_train)

,C,1.0
,kernel,'linear'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [39]:
y_pred=model.predict(X_test)

In [40]:
print("Classification Report:\n")
print(classification_report(y_test,y_pred))

Classification Report:

                precision    recall  f1-score   support

   bad weather       0.98      0.92      0.95       102
     collision       0.99      0.94      0.97       108
fire/explosion       0.97      0.87      0.92       120
     foundered       0.99      1.00      0.99      1024
 iceberg wreck       0.00      0.00      0.00        10
         mined       0.70      0.68      0.69        66
  naval battle       1.00      0.98      0.99        56
   ran aground       0.97      0.92      0.94       162
          sank       0.92      0.97      0.95       180
       scuttle       0.87      0.96      0.91        49
     torpedoed       1.00      0.98      0.99       190
       wrecked       0.95      1.00      0.97       613

      accuracy                           0.97      2680
     macro avg       0.86      0.85      0.86      2680
  weighted avg       0.96      0.97      0.96      2680



In [41]:
print("Accuracy Score: \n")
print(accuracy_score(y_test,y_pred))

Accuracy Score: 

0.9656716417910448


In [46]:
from sklearn.neighbors import KNeighborsClassifier


In [48]:
model=KNeighborsClassifier()


In [49]:
model.fit(X_train,y_train)

,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [50]:
y_pred=model.predict(X_test)

In [51]:
print("Classification Report:\n")
print(classification_report(y_test,y_pred))

Classification Report:

                precision    recall  f1-score   support

   bad weather       0.90      0.34      0.50       102
     collision       0.97      0.59      0.74       108
fire/explosion       0.45      0.62      0.53       120
     foundered       0.99      0.93      0.96      1024
 iceberg wreck       0.00      0.00      0.00        10
         mined       0.09      0.88      0.16        66
  naval battle       1.00      1.00      1.00        56
   ran aground       0.94      0.40      0.56       162
          sank       0.90      0.42      0.58       180
       scuttle       0.75      0.43      0.55        49
     torpedoed       0.97      0.61      0.75       190
       wrecked       0.96      0.67      0.79       613

      accuracy                           0.72      2680
     macro avg       0.74      0.57      0.59      2680
  weighted avg       0.92      0.72      0.78      2680



C:\Users\drapa\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\drapa\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\drapa\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [52]:
print("Accurracy Score:\n" )
print(accuracy_score(y_test,y_pred))

Accurracy Score:

0.7179104477611941
